# Metrics

1. Abnormal returns
2. Standardized Unexpected Earnings (SUE)
3. Portfolio Assignment
4. Continuously balanced SUE

### **3.2.1. Abnormal Returns**
We first load CRSP global NYSE data and selected firms data to measure abnormal returns. Abnormal returns are measured as:

$$ \begin{aligned}
&AR_{jt}=R_{jt}-R_{pt} \\
&AR_{jt}=\text{Abnormal return for firm j, day t} \\
&R_{jt}\ \ \ =\text{Raw return for j, day t} \\
&R_{pt}\ \ \ =\text{Equally weighted mean return for day t on the NYSE} \\
&\qquad \ \ \  \ \ \   \text{firm size (market value of common equity) decile that} \\
&\qquad \ \ \  \ \ \ \text{j is a part of at the beginning of the calendar year}
\end{aligned}$$

We use *DlyCap* (in thousand of dollars) as a proxy for firm size and define deciles for each calendar year.

In [1]:
import pandas as pd

# Loading data
dfD_NYSE = pd.read_csv("../data/OOS/dfD_NYSE.csv")
dfD_NYSE['DlyCalDt'] = pd.to_datetime(dfD_NYSE['DlyCalDt'])
dfD_NYSE['Year'] = dfD_NYSE['DlyCalDt'].dt.year

We define the first trading day of each year before assigning market cap deciles for each given year.

In [2]:
# First trading day per year
first_trading_day = (
    dfD_NYSE.groupby('Year')['DlyCalDt']
    .min()
    .reset_index()
    .rename(columns={'DlyCalDt': 'FirstTradingDay'})
)

# DlyCap on first trading day for each firm and assign size deciles to each firm for that year
jan_cap = dfD_NYSE.merge(
    first_trading_day[['Year', 'FirstTradingDay']],
    left_on=['Year', 'DlyCalDt'],
    right_on=['Year', 'FirstTradingDay']
)[['PERMNO', 'Year', 'DlyCap']]

jan_cap['SizeDecile'] = jan_cap.groupby('Year')['DlyCap'].transform(
    lambda x: pd.qcut(x, 10, labels=False, duplicates='drop') + 1
)

# Attach size decile to every daily observation (Year + PERMNO)
dfD_NYSE = dfD_NYSE.merge(jan_cap[['PERMNO', 'Year', 'SizeDecile']], on=['PERMNO', 'Year'], how='left')

# Equally weighted mean return per decile per day (R_pt)
decile_returns = (
    dfD_NYSE.groupby(['DlyCalDt', 'SizeDecile'])['DlyRet']
    .mean()
    .reset_index()
    .rename(columns={'DlyRet': 'R_decile'})
)

In [3]:
dfD = pd.read_csv("../data/OOS/dfD_final.csv")
dfD['DlyCalDt'] = pd.to_datetime(dfD['DlyCalDt'])
dfD['Year'] = dfD['DlyCalDt'].dt.year

# Drop existing sizeDecile
dfD = dfD.drop(columns=[c for c in dfD.columns if 'SizeDecile' in c or 'R_decile' in c or 'DlyAbnReturn' in c], errors='ignore')

# Assign full-NYSE size decile to each firm in dfD_final
dfD = dfD.merge(jan_cap[['PERMNO', 'Year', 'SizeDecile']], on=['PERMNO', 'Year'], how='left')

# Merge benchmark return (R_pt) and compute abnormal return
dfD = dfD.merge(decile_returns, on=['DlyCalDt', 'SizeDecile'], how='left')
dfD['DlyAbnReturn'] = dfD['DlyRet'] - dfD['R_decile']

# Check
print(dfD[['PERMNO', 'DlyCalDt', 'DlyRet', 'SizeDecile', 'R_decile', 'DlyAbnReturn']].head())
print(f"\nMissing DlyAbnReturn: {dfD['DlyAbnReturn'].isna().sum()}\nTotal rows: {len(dfD)}")

   PERMNO   DlyCalDt    DlyRet  SizeDecile  R_decile  DlyAbnReturn
0   10028 2012-01-03 -0.001339         2.0  0.014714     -0.016053
1   10028 2012-01-04 -0.010724         2.0  0.000531     -0.011255
2   10028 2012-01-05 -0.027100         2.0  0.003413     -0.030513
3   10028 2012-01-06  0.012535         2.0  0.001570      0.010965
4   10028 2012-01-09 -0.019257         2.0  0.007874     -0.027131

Missing DlyAbnReturn: 149837
Total rows: 6076990


We are missing quite a few DlyAbnReturn. This can likely be explained by the fact that we don't handle cases of firms not present on the NYSE/AMEX on the first trading day of the year

In [4]:
# For firms missing a decile for a given year, use their first available DlyCap in that year instead of earliest Jan
first_cap_per_firm_year = (
    dfD_NYSE[dfD_NYSE['DlyCap'].notna()]
    .sort_values('DlyCalDt')
    .groupby(['PERMNO', 'Year'])
    .first()
    .reset_index()[['PERMNO', 'Year', 'DlyCap']]
)

# Recompute deciles on this expanded table
first_cap_per_firm_year['SizeDecile'] = first_cap_per_firm_year.groupby('Year')['DlyCap'].transform(
    lambda x: pd.qcut(x, 10, labels=False, duplicates='drop') + 1
)

# Use this as fallback for firms missing decile
dfD = dfD.merge(
    first_cap_per_firm_year[['PERMNO', 'Year', 'SizeDecile']].rename(columns={'SizeDecile': 'SizeDecile_fallback'}),
    on=['PERMNO', 'Year'], how='left'
)

dfD['SizeDecile'] = dfD['SizeDecile'].fillna(dfD['SizeDecile_fallback'])
dfD = dfD.drop(columns='SizeDecile_fallback')
dfD = dfD[dfD['SizeDecile'].notna()] # 1012 rows that don't have any DlyCap over the year

# Recompute AbnReturns with filled deciles
dfD = dfD.merge(decile_returns, on=['DlyCalDt', 'SizeDecile'], how='left', suffixes=('_old', ''))
dfD = dfD.drop(columns=[c for c in dfD.columns if c.endswith('_old')])
dfD['DlyAbnReturn'] = dfD['DlyRet'] - dfD['R_decile']


print(f"NaN DlyAbnReturn after Jan-fix fallback: {dfD['DlyAbnReturn'].isna().sum()} out of {len(dfD)} rows")

# We save this version with computed DlyAbnReturns
dfD.to_csv("../data/OOS/dfD_final_Abn.csv", index=False)

NaN DlyAbnReturn after Jan-fix fallback: 1628 out of 6076990 rows


We are able to get back about 103 183 daily returns this way. Remaining missing values come from missing Closing price values.

The paper mentions that "*Observations were excluded from the analysis if the return for the earnings announcement day was missing on CRSP, or if the CRSP returns series did not encompass the 160 trading days surrounding the earnings announcement*". We hence exclude:
1. Quarters for which the return for the earnings announcement day is missing
2. When the CRSP returns series do not respect the 160 trading days window. *I imagine that 160 trading days corresponds to a ±80 days symmetrical window, but they after that they only mention a ±60 days window in the paper. I also interpreted that we only validate 160 trading days with continuous AbnormalReturns*

In [5]:
dfQ_final = pd.read_csv("../data/OOS/dfQ_final.csv")
dfQ_final['rdq'] = pd.to_datetime(dfQ_final['rdq'])

dfQ_final = dfQ_final.merge(
    dfD[['PERMNO', 'DlyCalDt', 'DlyAbnReturn']],
    left_on=['PERMNO', 'rdq'],
    right_on=['PERMNO', 'DlyCalDt'],
    how='left'
)

# Exclude if return missing on announcement day
before = len(dfQ_final)
dfQ_final = dfQ_final[dfQ_final['DlyAbnReturn'].notna()]
print(f"Returns on earnings announcement day missing: Dropped {before - len(dfQ_final)} firm-quarters, {len(dfQ_final)} remaining")

Returns on earnings announcement day missing: Dropped 17929 firm-quarters, 93316 remaining


In [6]:
# For each PERMNO, get sorted array of trading days with valid returns
trading_days = (
    dfD[dfD['DlyAbnReturn'].notna()]
    .groupby('PERMNO')['DlyCalDt']
    .apply(lambda x: sorted(x))
    .reset_index()
    .rename(columns={'DlyCalDt': 'TradingDays'})
)

dfQ_final = dfQ_final.merge(trading_days, on='PERMNO', how='left')

# For each firm-quarter, find index of announcement date and check 80 trading days on each side
def check_160_trading_days(row):
    dates = row['TradingDays']
    rdq = row['rdq']
    if rdq not in dates:
        return False
    idx = dates.index(rdq)
    return (idx >= 80) and (idx + 80 < len(dates)) 

before = len(dfQ_final)
dfQ_final['TradingDays'] = dfQ_final['TradingDays'].apply(list)
mask = dfQ_final.apply(check_160_trading_days, axis=1)
dfQ_final = dfQ_final[mask].drop(columns='TradingDays')
print(f"160-days window missing: Dropped {before - len(dfQ_final)} firm-quarters, {len(dfQ_final)} remaining")

160-days window missing: Dropped 5159 firm-quarters, 88157 remaining


We have 85957 firm-quarters using epspxq. This is slightly above the 84792 used by the authors. We need to check with the other EPS, and with some other cleaning methods.

### **3.2.2. Estimation of standardized unexpected earnings (SUE)**

"*Earnings were forecasted by estimating the Foster (1977) model with historical data. [The Foster model assumes that earnings follow a first-order autoregressive process in seasonal differences. FOS indicate that they used a maximum of 20 observations to estimate the Foster model. We used a maximum of 24 observations. FOS indicate that firms were included in the sample even if only ten consecutive quarters of data were available. We retained such firms also, but where fewer than 16 observations were available, we assumed that earnings followed a seasonal random walk.]*" 

"*The difference between actual and forecasted earnings was then scaled by the standard deviation of forecast errors over the estimation period to obtain standardized unexpected earnings or *SUE*.*"


**Data inclusion critera**
- Maximum of 24 observation to estimate the regression
- If 10 < quarters < 16, we retain firms but assume that earnings followed a seasonal random walk.
- We scale the residual by the Sd of forecast errors over the estimation period


Min: 10 quarters. Max: 24 quarters.

We first load the entire Compustat history, check past consecutive quarters (notably before 1974) for firms shortlisted from the 1974-1986 window. We then precompute Foster regressors.

In [7]:
# Loading entire Compustat (1999-2025)
dfQ_full = pd.read_csv("../data/WRDS/Compustat_Q_199903_202512.csv", low_memory=False)
dfQ_full['datadate'] = pd.to_datetime(dfQ_full['datadate'])

# Fix 3
dfQ_full = dfQ_full.sort_values(['gvkey', 'fyearq', 'fqtr', 'datadate'])
dfQ_full = dfQ_full.drop_duplicates(subset=['gvkey', 'fyearq', 'fqtr'], keep='last')
dfQ_full = dfQ_full.drop_duplicates(subset=['gvkey', 'datadate'], keep='last')

# Checking history for firms selected from the 2012-2024 window
# Filter in int then convert to str
shortlisted_gvkeys_int = set(int(g) for g in dfQ_final['gvkey'].unique())
dfQ_full = dfQ_full[dfQ_full['gvkey'].isin(shortlisted_gvkeys_int)]
dfQ_full['gvkey'] = dfQ_full['gvkey'].astype(str)
print(f"After filter: {dfQ_full.shape}")

dfQ_full = dfQ_full.sort_values(['gvkey', 'fyearq', 'fqtr']).reset_index(drop=True)
dfQ_full = dfQ_full.dropna(subset=['epspxq'])
print(f"After dropna epspxq: {dfQ_full.shape}")

results = []
seen = set()

After filter: (191693, 18)
After dropna epspxq: (185525, 18)


To optimize the computation of SUE on all firm-quarters, we pre calculate needed metrics on a global basis.

In [8]:
# Quarter index
dfQ_full['pidx'] = dfQ_full['fyearq'] * 4 + dfQ_full['fqtr']

# Precompute lags per firm
g = dfQ_full.groupby('gvkey')['epspxq']
dfQ_full['eps_l1'] = g.shift(1)
dfQ_full['eps_l4'] = g.shift(4)
dfQ_full['eps_l5'] = g.shift(5)

# Foster regressors
dfQ_full['y_foster'] = dfQ_full['epspxq'] - dfQ_full['eps_l4']
dfQ_full['x_foster'] = dfQ_full['eps_l1'] - dfQ_full['eps_l5']

# pidx lags for consecutiveness check
dfQ_full['pidx_l1'] = dfQ_full.groupby('gvkey')['pidx'].shift(1)
dfQ_full['pidx_l4'] = dfQ_full.groupby('gvkey')['pidx'].shift(4)
dfQ_full['pidx_l5'] = dfQ_full.groupby('gvkey')['pidx'].shift(5)

# Check first firm
gvkey_0, firm_0 = next(iter(dfQ_full.groupby('gvkey')))
firm_0 = firm_0.reset_index(drop=True)
print(f"\nFirst firm {gvkey_0}: {len(firm_0)} quarters")
print(f"Date range: {firm_0['datadate'].min()} to {firm_0['datadate'].max()}")
print(firm_0[['datadate', 'pidx', 'epspxq', 'y_foster', 'x_foster']].head())


First firm 10000: 108 quarters
Date range: 1999-03-31 00:00:00 to 2025-12-31 00:00:00
    datadate  pidx  epspxq  y_foster  x_foster
0 1999-03-31  7997    0.28       NaN       NaN
1 1999-06-30  7998    0.92       NaN       NaN
2 1999-09-30  7999    0.80       NaN       NaN
3 1999-12-31  8000   -1.36       NaN       NaN
4 2000-03-31  8001    0.03     -0.25       NaN


We then iterate on each firm, identified with its gvkey. We respect the 10 minimum required quarters, the maximum 24 quarters used to estimate the regression, and the seasonal random walk for firms with 10 to 16 quarters.

In [9]:
import numpy as np

for gvkey, firm in dfQ_full.groupby('gvkey'):
    firm = firm.reset_index(drop=True)

    for i in range(len(firm)):
        event_row = firm.iloc[i]

        # Only compute SUE for event quarters in 2012-2024
        if not (pd.Timestamp('2012-01-01') <= event_row['datadate'] <= pd.Timestamp('2024-12-31')): continue

        # Fix 4
        # Skip if this (gvkey, datadate) was already appended (fiscal year redefinition duplicates)
        if (gvkey, event_row['datadate']) in seen: continue

        # History: max 24, min 10
        hist = firm.iloc[max(0, i-24):i].copy().reset_index(drop=True)
        if len(hist) < 10: continue

        # Keep only the last consecutive block
        diffs = hist['pidx'].diff()
        breaks = diffs[diffs != 1].index.tolist()
        if breaks:
            hist = hist.iloc[breaks[-1]:].reset_index(drop=True)
        if len(hist) < 10: continue

        # Extract lagged EPS values
        q_last  = hist.iloc[-1]['epspxq']   # Q_{t-1}
        q_last4 = hist.iloc[-4]['epspxq']   # Q_{t-4} (same quarter last year)
        q_last5 = hist.iloc[-5]['epspxq'] if len(hist) >= 5 else np.nan  # Q_{t-5}

        if pd.isna(q_last4): continue

        if len(hist) < 16: # Seasonal random walk (10-15 quarters of history)
            forecast = q_last4
            errors = hist['epspxq'].values[4:] - hist['epspxq'].values[:-4]
            sigma = errors.std(ddof=1)

        else: # If 16+quarters, Foster (1977): Q_t - Q_{t-4} = delta + phi*(Q_{t-1} - Q_{t-5})
            reg = hist[['y_foster', 'x_foster', 'pidx', 'pidx_l1', 'pidx_l4', 'pidx_l5']].dropna()
            reg = reg[
                (reg['pidx'] - reg['pidx_l1'] == 1) &
                (reg['pidx'] - reg['pidx_l4'] == 4) &
                (reg['pidx'] - reg['pidx_l5'] == 5)
            ]
            if len(reg) < 4: continue

            y = reg['y_foster'].values
            X = np.column_stack([np.ones(len(y)), reg['x_foster'].values])
            coefs, *_ = np.linalg.lstsq(X, y, rcond=None)
            delta, phi = coefs

            if pd.isna(q_last5):continue

            forecast  = q_last4 + delta + phi * (q_last - q_last5)
            residuals = y - X @ coefs
            sigma     = residuals.std(ddof=2)

        if sigma == 0 or pd.isna(sigma): continue

        # SUE = (actual - forecast) / sigma
        seen.add((gvkey, event_row['datadate'])) # Fix
        results.append({
            'gvkey'   : gvkey,
            'datadate': event_row['datadate'],
            'fyearq'  : event_row['fyearq'],
            'fqtr'    : event_row['fqtr'],
            'epspxq'  : event_row['epspxq'],
            'forecast': forecast,
            'sigma'   : sigma,
            'SUE'     : (event_row['epspxq'] - forecast) / sigma
        })

df_sue = pd.DataFrame(results)
print(f"{len(df_sue)} SUE observations computed")
if len(df_sue) > 0:
    print(df_sue['SUE'].describe())

97683 SUE observations computed
count    97683.000000
mean         0.028919
std         80.643695
min     -14271.210313
25%         -0.479055
50%          0.009770
75%          0.497014
max      20747.821503
Name: SUE, dtype: float64


We merge this dataset with filtered firm-quarters to filter out incorrect quarters.

In [10]:
df_sue['gvkey'] = df_sue['gvkey'].astype(str)
df_sue['datadate'] = pd.to_datetime(df_sue['datadate'])

dfQ_final['gvkey'] = dfQ_final['gvkey'].astype(str)
dfQ_final['datadate'] = pd.to_datetime(dfQ_final['datadate'])

dfQ_final = dfQ_final.drop_duplicates(subset=['gvkey', 'datadate'], keep='first')

df_sue_final = df_sue.merge(
    dfQ_final[['gvkey', 'datadate', 'PERMNO', 'rdq']],
    on=['gvkey', 'datadate'],
    how='inner'
)

df_sue_final.to_csv("../data/OOS/SUE.csv", index=False)
print(f"{len(df_sue_final)} SUE after all filters")

83836 SUE after all filters


We have 84312 quarters: a bit less than expected, this might be because of diluted/basic EPS.

### **3.2.3. Portfolio assignment - hindisght bias**
"*Like FOS, we overcome that bias by assigning firms to portfolios on the basis of their standings relative to the distribution of unexpected earnings in the prior quarter*".

### **3.2.4. Continuously balanced SUE**

"*The continuously balanced SUE strategy works as follows. On a given   trading day, we identify any firms that announced earnings, and where standardized unexpected earnings fall in the upper quintile (good news) or lower quintile (bad news) of the prior-quarter distribution. If both good news and bad news firms exist for that day, we assume a long position in the former and a short position in the latter. The long (short) positions are initially equally weighted across the available good (bad) news firms, with the total amount of the long position exactly offsetting the total amount of the short position. We then compute buy-and-hold (i.e., continuously compounded) returns on each of the stocks in the long and short position, over the 60 trading days subsequent to the earnings announcement*."



"*On 14% of trading days, there were either no new good news or no bad news firms available, and so no match could be created. In such cases, we "wait" until a match becomes available. For example, if two good news firms announced earnings on day 1, but no bad news firms announced, we would wait until at least one bad news firm announced earnings. If the first available bad news firm announced on day 4, it would be matched with all good news firms announcing from days 1 through 4, and we would then compound returns from day 5 through day 64.In 97% of all cases, a match became available within two days*."

"*To provide some control for the Banz-Reinganum size effect, this matching process was always conducted within groups of small, medium, and large firms. Small firms are those whose January 1 market value of equity was among the lowest four deciles of the NYSE/AMEX, whereas large firms are those among the highest three deciles. Using only three size groups increased the probability of finding matches of good news and bad news firms within a short period of time. Since we used only three size groups (versus ten in the FOS control portfolio approach), our control for size is not as precise. However, if we assume that smaller firms are as likely to announce bad news as good news, this introduces no bias in the results.*"

**Steps**
- For every day, identify shortlisted firms that announced earnings
- Distribute prio-quarter SUE of every firms in quintiles
- We only keep firms for which SUE fell in the Q1 or Q5


If both Q1 and Q5 firms exist on that day:
- we assume a long-short position. Each leg of the position is equally weighted across available firms, with the long/short side perfectly offsetting each other.
- we compute continuously compounded buy&hold returns of each of the stocks of the position over the 60 days subsequent to the earnings announcement, without rebalancing

If both Q1 and Q5 firms aren't available, we wait until a match becomes available:
- If the first available bad news firm announced on day 4, it would be matched with all good news firms announcing from days 1 through 4, and we would then compound returns from day 5 through day 64
In 97% of all cases, a match became available within two days.

Before matching, we respect the logic of the paper by conducting the matching process among firms of similar size (small, medium, large). 
- small: Jan. 1st market cap: D1 to D4
- medium: Jan. 1st market cap: D5 to D6
- large: Jan. 1st market cap: D7+

In [11]:
#### V2 that wasn't working: groupsize then Q1/Q5

## Assign size group to each firm-quarter using January size decile of announcement year
#df_sue_final['rdq'] = pd.to_datetime(df_sue_final['rdq'])
#df_sue_final['ann_year'] = df_sue_final['rdq'].dt.year
#
#df_sue_final = df_sue_final.merge(
#    jan_cap[['PERMNO', 'Year', 'SizeDecile']],
#    left_on=['PERMNO', 'ann_year'],
#    right_on=['PERMNO', 'Year'],
#    how='left'
#)
#
## Assign size group
#df_sue_final['SizeGroup'] = pd.cut(
#    df_sue_final['SizeDecile'],
#    bins=[0, 4, 7, 10],
#    labels=['small', 'medium', 'large']
#)
#
#df_sue_final.to_csv("../data/SUE.csv", index=False)  # save with SizeGroup on 84k
#print(df_sue_final['SizeGroup'].value_counts())
#print(f"Missing SizeGroup: {df_sue_final['SizeGroup'].isna().sum()}")
#
## We identify prior quarter for each firm-quarter
#df_sue_final['prev_fyearq'] = np.where(df_sue_final['fqtr'] == 1, df_sue_final['fyearq'] - 1, df_sue_final['fyearq'])
#df_sue_final['prev_fqtr']   = np.where(df_sue_final['fqtr'] == 1, 4, df_sue_final['fqtr'] - 1)
#
## We compute SUE quintile boundaries from prior-quarter distribution
#quintile_bounds = (
#    df_sue_final.groupby(['fyearq', 'fqtr'])['SUE']
#    .quantile([0.2, 0.8])
#    .unstack()
#    .reset_index()
#    .rename(columns={0.2: 'Q20', 0.8: 'Q80', 'fyearq': 'prev_fyearq', 'fqtr': 'prev_fqtr'})
#)
#
## We merge bounds onto each firm-quarter via its prior quarter
#df_sue_final = df_sue_final.merge(quintile_bounds, on=['prev_fyearq', 'prev_fqtr'], how='left')
#
## We assign quintile label — only Q1 (bad news) and Q5 (good news) are kept
#df_sue_final['SUE_quintile'] = np.where(df_sue_final['SUE'] <= df_sue_final['Q20'], 1,
#                               np.where(df_sue_final['SUE'] >= df_sue_final['Q80'], 5, np.nan))
#
#before = len(df_sue_final)
#df_sue_final = df_sue_final[df_sue_final['SUE_quintile'].notna()]
#print(f"Kept {len(df_sue_final)} firm-quarters in Q1 or Q5 (dropped {before - len(df_sue_final)})")
#print(df_sue_final['SUE_quintile'].value_counts().sort_index())

Because of compatibility issue between proper FIG2 and GroupSize for all quarters, we first separately compute GroupSize for all firms, with no Q1/Q5 filtering, and export it as SUE_sizegroup.csv

In [12]:
# Compute SizeGroup on full 84k and save (for size-group graphs)
df_sue_full = pd.read_csv("../data/OOS/SUE.csv")
df_sue_full['rdq'] = pd.to_datetime(df_sue_full['rdq'])
df_sue_full['ann_year'] = df_sue_full['rdq'].dt.year
df_sue_full = df_sue_full.merge(
    jan_cap[['PERMNO', 'Year', 'SizeDecile']],
    left_on=['PERMNO', 'ann_year'],
    right_on=['PERMNO', 'Year'],
    how='left'
)
df_sue_full['SizeGroup'] = pd.cut(
    df_sue_full['SizeDecile'],
    bins=[0, 4, 7, 10],
    labels=['small', 'medium', 'large']
)
# Save with SizeGroup on 84k (used for CAR graphs by size)
df_sue_full[['gvkey', 'fyearq', 'fqtr', 'SizeGroup']].to_csv("../data/OOS/SUE_sizegroup.csv", index=False)
print(df_sue_full['SizeGroup'].value_counts())

SizeGroup
large     34508
medium    27303
small     21671
Name: count, dtype: int64


We then use the previous correct order: we first compute SUE quintile boundaries from prior-quarter distribution, then attribute group sizes. We use this one for the FIG2.

In [13]:
### Q1/Q5
# We identify prior quarter for each firm-quarter
df_sue_final['prev_fyearq'] = np.where(df_sue_final['fqtr'] == 1, df_sue_final['fyearq'] - 1, df_sue_final['fyearq'])
df_sue_final['prev_fqtr']   = np.where(df_sue_final['fqtr'] == 1, 4, df_sue_final['fqtr'] - 1)

# We compute SUE quintile boundaries from prior-quarter distribution
quintile_bounds = (
df_sue_final.groupby(['fyearq', 'fqtr'])['SUE']
    .quantile([0.2, 0.8])
    .unstack()
    .reset_index()
    .rename(columns={0.2: 'Q20', 0.8: 'Q80', 'fyearq': 'prev_fyearq', 'fqtr': 'prev_fqtr'})
)

# We merge bounds onto each firm-quarter via its prior quarter
df_sue_final = df_sue_final.merge(quintile_bounds, on=['prev_fyearq', 'prev_fqtr'], how='left')

# We assign quintile label — only Q1 (bad news) and Q5 (good news) are kept
df_sue_final['SUE_quintile'] = np.where(df_sue_final['SUE'] <= df_sue_final['Q20'], 1,
np.where(df_sue_final['SUE'] >= df_sue_final['Q80'], 5, np.nan))

before = len(df_sue_final)
df_sue_final = df_sue_final[df_sue_final['SUE_quintile'].notna()]

print(f"Kept {len(df_sue_final)} firm-quarters in Q1 or Q5 (dropped {before - len(df_sue_final)})")
print(df_sue_final['SUE_quintile'].value_counts().sort_index())


### SizeGroup
## Assign size group to each firm-quarter using January size decile of announcement year
df_sue_final['rdq'] = pd.to_datetime(df_sue_final['rdq'])
df_sue_final['ann_year'] = df_sue_final['rdq'].dt.year
df_sue_final = df_sue_final.merge(
jan_cap[['PERMNO', 'Year', 'SizeDecile']],
left_on=['PERMNO', 'ann_year'],
right_on=['PERMNO', 'Year'],
how='left'
)

# Assign size group
df_sue_final['SizeGroup'] = pd.cut(
df_sue_final['SizeDecile'],

bins=[0, 4, 7, 10],
labels=['small', 'medium', 'large']
)

print(f"\n{df_sue_final['SizeGroup'].value_counts()}")
print(f"Missing SizeGroup: {df_sue_final['SizeGroup'].isna().sum()}")
df_sue_final.to_csv('../data/OOS/SUE_Q1Q5_size.csv')


Kept 33698 firm-quarters in Q1 or Q5 (dropped 50138)
SUE_quintile
1.0    16764
5.0    16934
Name: count, dtype: int64

SizeGroup
large     14212
medium    10840
small      8465
Name: count, dtype: int64
Missing SizeGroup: 185


We then proceed with the matching.

In [14]:
# We get sorted list of unique trading days
trading_days_list = sorted(dfD['DlyCalDt'].unique())
day_to_idx = {day: idx for idx, day in enumerate(trading_days_list)}  

# For each trading day for which firms announced earnings, with respect to size group
announcements = df_sue_final[['gvkey', 'PERMNO', 'rdq', 'SUE_quintile', 'SizeGroup']].copy()
announcements['rdq'] = pd.to_datetime(announcements['rdq'])
announcements = announcements[announcements['SizeGroup'].notna()]

results_strategy, all_wait_days = [], []
total_days_with_ann, total_days_no_match = 0, 0


for size_group in ['small', 'medium', 'large']:
    ann_group = announcements[announcements['SizeGroup'] == size_group]
    pending_good, pending_bad = [], []
    pending_since = None

    for day in trading_days_list:
        today    = ann_group[ann_group['rdq'] == day]
        new_good = today[today['SUE_quintile'] == 5][['PERMNO', 'rdq']].to_dict('records')
        new_bad  = today[today['SUE_quintile'] == 1][['PERMNO', 'rdq']].to_dict('records')

        had_announcement = len(new_good) > 0 or len(new_bad) > 0
        pending_good.extend(new_good)
        pending_bad.extend(new_bad)

        if had_announcement and pending_since is None:
            pending_since = day

        if not pending_good or not pending_bad:
            if had_announcement:
                total_days_with_ann += 1
                total_days_no_match += 1
            continue

        if had_announcement:
            total_days_with_ann += 1

        day_idx = day_to_idx[day]
        if day_idx + 1 >= len(trading_days_list) or day_idx + 61 >= len(trading_days_list): continue

        wait = day_to_idx[day] - day_to_idx[pending_since] if pending_since != day else 0
        all_wait_days.append(wait)

        results_strategy.append({
            'size_group' : size_group,
            'match_day'  : day,
            'start_day'  : trading_days_list[day_idx + 1],
            'end_day'    : trading_days_list[day_idx + 61],
            'good_firms' : [f['PERMNO'] for f in pending_good],
            'bad_firms'  : [f['PERMNO'] for f in pending_bad],
            'n_good'     : len(pending_good),
            'n_bad'      : len(pending_bad),
            'wait_days'  : wait,
        })

        pending_good, pending_bad = [], []
        pending_since = None

df_strategy = pd.DataFrame(results_strategy)
wait_series = pd.Series(all_wait_days)
print(f"{len(df_strategy)} matched positions created")
print(df_strategy.groupby('size_group')[['n_good', 'n_bad']].mean().round(2))

3960 matched positions created
            n_good  n_bad
size_group               
large         4.99   4.92
medium        4.27   4.34
small         3.42   3.26


We compute statistics of interest (14% and 97% in the corpus).

In [15]:
print(f"Days with no match      : {total_days_no_match/total_days_with_ann*100:.1f}%")
print(f"Match within 2 days     : {(wait_series <= 1).mean()*100:.1f}%")

Days with no match      : 34.4%
Match within 2 days     : 82.2%


Without group matching: For days we no available match, we find 12% instead of the announced 14%. This can probably be fixed by reaching the correct amount of earnings announcement. For 2 days match, we find 97%, just like the paper.

With group matching: Days with no match: 32%, Match within 2 days: 86.6%

From these matched positions, we can compute returns, with equally-weigthed positions offseting the initial value of the other leg. We normalize to $1 long / $1 short, equally weighted within each leg, with each good firm receiving 1/n_good, each bad firm getting 1/n_bad.
We vectorize it to treat our large dataset.

In [16]:
# Pre-index dfD by PERMNO for fast lookup
dfD_indexed = dfD.set_index(['PERMNO', 'DlyCalDt']).sort_index()

# Explode df_strategy into one row per firm
long_rows = df_strategy.explode('good_firms').rename(columns={'good_firms': 'PERMNO'})
long_rows['leg'] = 'long'
short_rows = df_strategy.explode('bad_firms').rename(columns={'bad_firms': 'PERMNO'})
short_rows['leg'] = 'short'

positions = pd.concat([
    long_rows[['match_day', 'start_day', 'end_day', 'size_group', 'PERMNO', 'leg']],
    short_rows[['match_day', 'start_day', 'end_day', 'size_group', 'PERMNO', 'leg']]
]).reset_index(drop=True)

# Merge with dfD to get all daily returns for each position
positions['PERMNO'] = positions['PERMNO'].astype(dfD['PERMNO'].dtype)
dfD_small = dfD[['PERMNO', 'DlyCalDt', 'DlyRet']].copy()

positions = positions.merge(dfD_small, on='PERMNO', how='left')
positions = positions[
    (positions['DlyCalDt'] >= positions['start_day']) &
    (positions['DlyCalDt'] <= positions['end_day'])
]

# Buy-and-hold return per firm per position
positions = positions.dropna(subset=['DlyRet'])
bh_returns = (
    positions.groupby(['match_day', 'size_group', 'leg', 'PERMNO'])['DlyRet']
    .apply(lambda x: (1 + x).prod() - 1)
    .reset_index()
    .rename(columns={'DlyRet': 'bh_return'})
)

# Equally weighted per leg
leg_returns = (
    bh_returns.groupby(['match_day', 'size_group', 'leg'])['bh_return']
    .mean()
    .unstack()
    .reset_index()
    .rename(columns={'long': 'r_long', 'short': 'r_short'})
)

leg_returns['r_ls'] = leg_returns['r_long'] - leg_returns['r_short']

print(f"{len(leg_returns)} positions with computed returns")
print(leg_returns[['r_long', 'r_short', 'r_ls']].describe())
print(f"\nMean long-short return: {leg_returns['r_ls'].mean():.4f}")

# We save
leg_returns.to_csv("../data/OOS/cont_SUE.csv", index=False)

3960 positions with computed returns
leg         r_long      r_short         r_ls
count  3960.000000  3960.000000  3960.000000
mean      0.052231     0.015235     0.036995
std       0.228531     0.192665     0.261165
min      -0.941388    -0.941388    -2.455127
25%      -0.040899    -0.078775    -0.064246
50%       0.037312     0.011322     0.025611
75%       0.119972     0.095429     0.128315
max       7.066215     2.356407     7.030423

Mean long-short return: 0.0370


We are hence able to compute the continuously balanced SUE strategy.